# Workshop 4 — best8: restauración y anomalías sin entrenamiento

Este notebook tiene dos operaciones de lectura: inspeccionar el best.pt real de
Drive y evaluar anomalías con sus pesos congelados. **No contiene una celda de
entrenamiento ni restaura Adam durante la evaluación de anomalías.**

La ventana weights_only comunicada (160.40 → 227.15 → 193.81; 16 updates) usó
Adam nuevo y otro prefijo de datos: no reprodujo el entrenamiento histórico.
`continue` de fase 4 solo reanuda su propio formato/cursor/prefijo. No se debe
renombrar best.pt ni cambiarle el rol para eludir esa restricción.

Primero comprobar integridad y correspondencia temporal de pesos, Adam, RNG,
configuración e historia. Tener exp_avg/exp_avg_sq no demuestra que pertenezcan
a los pesos de época 8. Una restauración mecánica sin pasos tampoco prueba
fidelidad al backend y al recorrido histórico. No se prepara otra continuación
ejecutable mientras esa fidelidad permanezca sin demostrar.


In [ ]:
from google.colab import drive, files
drive.mount("/content/drive")
from pathlib import Path
from datetime import datetime, timezone
import json, subprocess, sys, zipfile
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")


In [ ]:
# @title Rutas confirmadas
HISTORICAL_RUN = "/content/drive/MyDrive/Workshop_4/runs/workshop4_clipping_v2_prueba12/ae_z32" # @param {type:"string"}
PROCESSED_DIR = "/content/drive/MyDrive/Workshop_4/data/processed" # @param {type:"string"}
OUTPUT_PARENT = "/content/drive/MyDrive/Workshop_4/best8_restore_anomalies" # @param {type:"string"}
ZIP_SOURCE = "Upload" # @param ["Upload", "Drive"]
PROJECT_ZIP = "/content/drive/MyDrive/Workshop_4/workshop4_colab_best8_anomalies.zip" # @param {type:"string"}
EXTRACT_DIR = "/content/workshop4_best8_anomalies" # @param {type:"string"}
historical = Path(HISTORICAL_RUN)
BEST = historical / "checkpoints/best.pt"
LAST = historical / "checkpoints/last.pt"
METADATA = historical / "metadata.json"
print(json.dumps({"best8": BEST.is_file(), "last30": LAST.is_file(),
                  "metadata": METADATA.is_file(),
                  "train": (Path(PROCESSED_DIR)/"train.parquet").is_file(),
                  "val": (Path(PROCESSED_DIR)/"val.parquet").is_file()}, indent=2))
if not BEST.is_file():
    raise FileNotFoundError("No hay best.pt en la ruta indicada después de montar Drive")


In [ ]:
if ZIP_SOURCE == "Upload":
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Subir exactamente el ZIP nuevo best8/anomalías")
    PROJECT_ZIP = next(iter(uploaded))
zip_path = Path(PROJECT_ZIP).expanduser().resolve()
with zipfile.ZipFile(zip_path) as archive:
    helpers = [n for n in archive.namelist() if n.endswith("Workshop_4/src/colab_utils.py")]
    if len(helpers) != 1:
        raise ValueError("ZIP de proyecto incompatible")
    zip_import_root = helpers[0].removesuffix("src/colab_utils.py")
sys.path.insert(0, str(zip_path) + "/" + zip_import_root)
from src.colab_utils import extract_project_zip, runtime_info
WORKSHOP_DIR = extract_project_zip(zip_path, EXTRACT_DIR)
assert (WORKSHOP_DIR/"src/checkpoint_fidelity.py").is_file()
assert (WORKSHOP_DIR/"src/anomaly_evaluation.py").is_file()
print(runtime_info())


## 1. Inspección del checkpoint real y Adam

La inspección usa `torch.load(weights_only=True)`, valida el contrato y compara
SHA antes/después. Revisa momentos/pasos por parámetro y su correspondencia con
history, además de configuración, RNG, clipping y manifest. Si last30 comparte
exactamente Adam/RNG con pesos distintos de best8, lo señala como evidencia de
mezcla temporal. Los pasos esperados solo se deducen de contadores de lotes
aceptados completos; no se inventan si faltan.

Un modelo/Adam desechables comprueban load_state_dict sin optimizer.step.
`structural_restore_possible=True` no autoriza reanudar. La inspección conserva
RNG/flags y no escribe en los originales. Opcionalmente descargar su JSON para
continuar el análisis local.


In [ ]:
RUN_CHECKPOINT_AUDIT = True # @param {type:"boolean"}
DOWNLOAD_AUDIT_JSON = False # @param {type:"boolean"}
AUDIT_DIR = Path(OUTPUT_PARENT) / f"checkpoint_audit_{run_id}"
audit_command = [sys.executable, "-m", "src.checkpoint_fidelity",
                 "--checkpoint", str(BEST), "--processed-dir", str(PROCESSED_DIR),
                 "--output-dir", str(AUDIT_DIR)]
if METADATA.is_file():
    audit_command += ["--metadata", str(METADATA)]
if LAST.is_file():
    audit_command += ["--comparison-checkpoint", str(LAST)]
subprocess.run(audit_command, cwd=WORKSHOP_DIR, check=True, timeout=30)
fidelity = None
if RUN_CHECKPOINT_AUDIT:
    subprocess.run(audit_command + ["--execute"], cwd=WORKSHOP_DIR, check=True, timeout=180)
    fidelity = json.loads((AUDIT_DIR/"report.json").read_text())
    display({key: fidelity.get(key) for key in
             ("availability", "epoch", "checkpoint_role", "adam", "temporal_alignment",
              "disposable_restore", "structural_restore_possible", "epoch_boundary_state_restorable",
              "data_bytes_reverified", "continuation_verdict", "historical_faithful_continuation_eligible")})
    print("No se inicia ninguna continuación de entrenamiento.")
    if DOWNLOAD_AUDIT_JSON:
        files.download(str(AUDIT_DIR/"report.json"))


## 2. Preparar anomalías con best8 congelado

Score principal fijado: MSE entre softmax y one-hot de x, promediado sobre
posiciones y vocabulario (no MSE entre IDs ordinales). Score auxiliar: NLL
categórica, en nats/secuencia. No se cambia el entrenamiento ni la arquitectura.

Normalidad: clase y de siguiente carácter más frecuente en TODO train (empates
por menor ID). Es una etiqueta proxy, no una anotación semántica de anomalía.
Se puede fijar normal_class en el JSON antes de evaluar, con soporte en train.

Los hablantes de val se dividen por hash/semilla en grupos disjuntos. Solo los
ejemplos normales del grupo de calibración fijan el umbral; el otro grupo es
evaluación de desarrollo mixta. Se muestrea mediante reservoir por hablante,
no por un prefijo adyacente. El programa registra cobertura y cuotas.

alpha=0.05 es un control prefijado, sin garantía de FPR nominal bajo cambios
de hablantes/ventanas correlacionadas. El umbral usa el orden
ceil((n+1)*(1-alpha)); si falta calibración suficiente se reporta indefinido.
Una puntuación igual al umbral queda normal. Los umbrales se guardan antes de
leer evaluación/test y no se ajustan a sus etiquetas.

**Limitación central:** el best8 general fue entrenado con todas las clases.
Sus métricas son exploratorias; no sustituyen el experimento normal-only que
exige el taller. Ese entrenamiento, VAE y sus variantes permanecen pendientes.


In [ ]:
# @title Plan de evaluación: cero entrenamiento
DEVICE = "cuda" # @param ["cuda", "cpu"]
RUN_ANOMALY_EVALUATION = False # @param {type:"boolean"}
EVALUATE_TEST = False # @param {type:"boolean"}
ANOMALY_DIR = Path(OUTPUT_PARENT) / f"anomaly_eval_{run_id}"
anomaly_command = [sys.executable, "-m", "src.anomaly_evaluation",
                   "--config", str(WORKSHOP_DIR/"config/anomaly_evaluation.json"),
                   "--checkpoint", str(BEST), "--processed-dir", str(PROCESSED_DIR),
                   "--device", DEVICE, "--output-dir", str(ANOMALY_DIR)]
if EVALUATE_TEST:
    anomaly_command.append("--evaluate-test")
subprocess.run(anomaly_command, cwd=WORKSHOP_DIR, check=True, timeout=30)


In [ ]:
if RUN_ANOMALY_EVALUATION:
    completed = subprocess.run(anomaly_command + ["--execute"], cwd=WORKSHOP_DIR,
                               check=False, timeout=650)
    print("Código de salida:", completed.returncode)
    if (ANOMALY_DIR/"summary.json").is_file():
        summary = json.loads((ANOMALY_DIR/"summary.json").read_text())
        print(summary["status"], summary.get("reason", ""))
        print("Tipo de detector:", summary.get("detector_status"))
        print("Calibración:", summary.get("calibration_status"))
        print("Pasos de optimizador:", summary["optimizer_steps"])
else:
    print("Solo se mostró el plan; no se calcularon puntuaciones ni se entrenó.")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
if RUN_ANOMALY_EVALUATION and (ANOMALY_DIR/"metrics.json").is_file():
    metrics = json.loads((ANOMALY_DIR/"metrics.json").read_text())
    rows = []
    for population, by_score in metrics.items():
        for score, m in by_score.items():
            rows.append({"población": population, "score": score, "n": m["examples"],
                         "AUC-ROC": m["roc_auc"], "AP": m["average_precision"],
                         "umbral": m["threshold"], "FPR": m["false_positive_rate"],
                         "precision": m["precision"], "recall": m["recall"], "F1": m["f1"]})
    display(pd.DataFrame(rows))
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for population, by_score in metrics.items():
        for score, m in by_score.items():
            roc, pr = m["roc_curve"], m["precision_recall_curve"]
            if roc:
                axes[0].plot([p["fpr"] for p in roc], [p["tpr"] for p in roc], label=f"{population}/{score}")
            if pr:
                axes[1].step([p["recall"] for p in pr], [p["precision"] for p in pr], where="post", label=f"{population}/{score}")
    axes[0].plot([0, 1], [0, 1], "k--", lw=.7)
    axes[0].set(xlabel="FPR", ylabel="TPR", title="ROC exploratoria")
    axes[1].set(xlabel="Recall", ylabel="Precision", title="Precision/recall exploratoria")
    for ax in axes:
        ax.grid(alpha=.2)
        ax.legend(fontsize=7)
    plt.tight_layout()
    plt.show()


## Límites e interpretación

Por defecto se puntúan hasta 2.048 normales de calibración y 2.048 ejemplos de
evaluación val, con cuotas por hablante. Test no se lee a menos que se active
EVALUATE_TEST; entonces se añaden hasta 2.048 ejemplos mixtos, con los umbrales
ya congelados. Son subconjuntos estratificados, no validación/test completos ni
los 128 ejemplos usados para diagnosticar reconstrucción histórica.

Los dos scores y su prioridad se fijan antes de test. No escoger score, clase
normal o umbral después de revisar los resultados de test. ROC-AUC es indefinido
cuando falta alguna clase; no se reemplaza por cero. Una calibración insuficiente
permite examinar ranking, pero no declarar un detector con umbral válido.

Conservar scores.csv, sampling.json, normal_definition.json, thresholds.json,
metrics.json, summary.json y el SHA del checkpoint. No se publica ninguna AUC
real hasta ejecutar la evaluación con el archivo histórico. Consultar
BEST8_RESTORE_AND_ANOMALIES.md antes de considerar otra continuación.
